# 1. Importando as Bibliotecas

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, mean_absolute_error, mean_squared_error
)

ModuleNotFoundError: No module named 'matplotlib'

# 2. Carregamento e Pré-processamento dos Dados

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (300, 300) 

print("Carregando base de Treino...")
train_dataset = image_dataset_from_directory('train', shuffle=True, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='binary')

print("Carregando base de Validação...")
valid_dataset = image_dataset_from_directory('valid', shuffle=True, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='binary')

print("Carregando base de Teste...")
test_dataset = image_dataset_from_directory('test', shuffle=False, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='binary')

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
valid_dataset = valid_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

# 3. Construção do Modelo (Transfer Learning)

In [ ]:
# Bloco de Data Augmentation (gira, inverte e dá zoom nas imagens durante o treino. pensei em fazer isso para aumentar a accuracy do modelo)
data_augmentation = tf.keras.Sequential([
    RandomFlip('horizontal_and_vertical'),
    RandomRotation(0.2),
    RandomZoom(0.2),
])

# Instanciando a EfficientNetB3
base_model = EfficientNetB3(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# Congelamos a base para a Fase 1
base_model.trainable = False

# Construindo a rede
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x) 
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.summary()

# 4. Treinamento - FASE 1 (Transfer Learning Padrão)

In [ ]:
print("Iniciando Fase 1: Treinando apenas a camada de saída...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fase1 = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=10
)

# 5. 5. Treinamento - FASE 2 (Fine-Tuning Profundo)

In [ ]:
print("Iniciando Fase 2: Fine-Tuning (Descongelando as últimas camadas)...")

# Descongelamento da rede base
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint('melhor_modelo_efficientnet_pc.keras', monitor='val_accuracy', save_best_only=True)
]

history_fase2 = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=15,
    callbacks=callbacks
)

# 6. Avaliação e Extração das Métricas

In [ ]:
print("A extrair predições do conjunto de Teste...")
y_true = np.concatenate([y for x, y in test_dataset], axis=0)

y_pred_probs = model.predict(test_dataset).ravel() 

y_pred = (y_pred_probs > 0.5).astype("int32") 

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

mae = mean_absolute_error(y_true, y_pred_probs)
rmse = np.sqrt(mean_squared_error(y_true, y_pred_probs))

print("\n" + "="*45)
print("📊 RESULTADOS FINAIS DAS MÉTRICAS")
print("="*45)
print(f"• Accuracy (Exatidão):        {accuracy:.4f}")
print(f"• Precision (Precisão):       {precision:.4f}")
print(f"• Recall (Revocação):         {recall:.4f}")
print(f"• F1-score:                   {f1:.4f}")
print(f"• ROC-AUC:                    {roc_auc:.4f}")
print(f"• MAE (Erro Médio Absoluto):  {mae:.4f}")
print(f"• RMSE (Raiz Erro Quadrático):{rmse:.4f}")
print("="*45 + "\n")

acc = history_fase1.history['accuracy'] + history_fase2.history['accuracy']
val_acc = history_fase1.history['val_accuracy'] + history_fase2.history['val_accuracy']
loss = history_fase1.history['loss'] + history_fase2.history['loss']
val_loss = history_fase1.history['val_loss'] + history_fase2.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Treino')
plt.plot(epochs_range, val_acc, label='Validação')
plt.axvline(x=len(history_fase1.history['accuracy'])-1, color='gray', linestyle='--', label='Início Fine-Tuning')
plt.title('Accuracy ao longo das Épocas')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Treino')
plt.plot(epochs_range, val_loss, label='Validação')
plt.axvline(x=len(history_fase1.history['loss'])-1, color='gray', linestyle='--', label='Início Fine-Tuning')
plt.title('Loss (Perda) ao longo das Épocas')
plt.legend(loc='upper right')
plt.show()

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Falha (0)', 'Sucesso (1)'], yticklabels=['Falha (0)', 'Sucesso (1)'])
plt.ylabel('Classe Real')
plt.xlabel('Predição do Modelo')
plt.title('Matriz de Confusão')

plt.subplot(1, 2, 2)
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC-AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC')
plt.legend(loc="lower right")

plt.tight_layout()
plt.show()